# 08 · Capstone: ShipFast and the research agent in LangGraph

Notebooks 01 to 07 built the pieces. This notebook puts them together into the two agents every
framework in this project implements against the same contract (`contracts/`), and shows how they are
evaluated. The agents live in `lg_agents/`, and `run_eval.py` runs the full evaluation with the same code.

**You will see**

- the ShipFast agent's design in LangGraph, and why its Resolution is built in code
- two hard ShipFast scenarios run live through the **shared runner**: an approval, and a two-turn case
- one research question through the shared research runner
- this framework's full results, and all three frameworks side by side

In [1]:
# The agent module, its research workflow, and the adapter the shared runner drives.
import lg_agents.shipfast as sf
from lg_agents.research_flow import answer
from lg_agents.shipfast import LangGraphShipFast

# The shared harness from contracts/: the model check, the two runners and the test data.
from agent_contracts.harness import verify_model
from agent_contracts.harness.runners import run_question, run_scenario
from agent_contracts.research.scoring import load_questions
from agent_contracts.shipfast.scoring import load_scenarios

verify_model()  # stops with the fix if Ollama is not running or the shared model is missing
# Index the scenarios and questions by id, so a cell can pick one by name.
scenarios = {s["id"]: s for s in load_scenarios()}
questions = {q["id"]: q for q in load_questions()}

## The ShipFast agent

In LangGraph the agent is a `StateGraph` (triage node, conditional edge, a fixed decline node, and the `create_agent`
specialist added as a subgraph with the approval middleware), with a checkpointer per conversation thread
that also holds a paused approval.

Every build fills in the scored `Resolution` the same way, with the shared
`agent_contracts.shipfast.resolution.derive_resolution` (DECISIONS.md 16): `intent` and `order_id` from
triage; `action`, `refund_amount` and `ticket_id` from what actually happened in the ledger and the
tickets; and only when a turn left no record does the model choose between `answered`, `needs_info` and
`declined`. The component notebooks showed why: asked to fill in a Resolution itself, the 8B model
claimed refunds that never happened and invented ticket numbers.

In [2]:
# The module docstring in lg_agents/shipfast.py describes the agent's design. It is shown here because
# the file is the one place that states the whole flow in a few lines.
print(sf.__doc__)

The ShipFast support agent in LangGraph (ShipFast SPEC.md), the capstone of notebooks 01 to 07.

    START -> triage --(intent == other)--> decline -> END
                    \-----(otherwise)----> specialist -> END

- triage: structured `Triage` from the conversation so far (shared `prompts.TRIAGE`).
- decline: a fixed reply, no model call (policy P7).
- specialist: the `create_agent` loop with the shared tools and instructions, added to the graph as a
  subgraph, with the $100 approval middleware (`interrupt()` before a large refund).
- A checkpointer keeps each conversation (thread) and holds a paused approval until it is resumed.

The Resolution is built by the shared `derive_resolution` from the ledger and tickets (DECISIONS.md 16);
the model only writes the reply and, when a turn left no record, picks the `Outcome`.



The docstring draws the graph. To check it against the real thing, ask the compiled graph for its edges. `__start__` and `__end__` are LangGraph's built-in entry and exit points, and a **conditional** edge is one the graph chooses at run time (here, by the triage intent).

In [3]:
# build_shipfast() compiles the graph (in lg_agents/shipfast.py); get_graph().edges lists its arrows.
print(f"{'from':11} {'to':11} kind")
for e in sf.build_shipfast().get_graph().edges:
    print(f"{e.source:11} {e.target:11} {'conditional' if e.conditional else 'always'}")

from        to          kind
__start__   triage      always
triage      decline     conditional
triage      specialist  conditional
decline     __end__     always
specialist  __end__     always


## Run through the shared runner

`run_scenario` is the same function every framework's `run_eval.py` uses. It resets the shared tools,
feeds each turn, resumes a paused approval with the scenario's decision, and checks the result against
`scenarios.json`. The framework only supplies the adapter: start a conversation, send a turn, resume.

`show()` turns a result into something you can read. It prints a summary block (verdict, cost and tools), a table with one row per customer turn (the `Resolution` the agent produced for that turn: intent, order, action and refund amount) with the start of the reply under it, and then any **problems**, which are the checks from `scenarios.json` that failed.

In [4]:
# show() prints one scenario result: a summary block, a table with one row per turn, then any problems.
def show(item):
    print(f"Scenario:    {item['id']}")
    print(f"Result:      {'PASS' if item['passed'] else 'FAIL'}")  # PASS means every check in scenarios.json held
    print(f"Cost:        {item['llm_calls']} model calls in {item['latency_s']:.0f}s")
    print(f"Tools used:  {', '.join(item['tool_calls']) or 'none'}")  # in the order the agent called them
    print()
    print(f"{'Turn':4}  {'Intent':15} {'Order':6} {'Action':11} {'Refund':>7}")
    for i, t in enumerate(item["turn_resolutions"], 1):
        refund = t["refund_amount"] or "-"  # "-" when no money moved
        print(f"{i:<4}  {t['intent']:15} {t['order_id'] or '-':6} {t['action']:11} {refund:>7}")
        print(f"      reply: {t['reply'].strip()[:160]}")  # the first 160 characters of what the customer saw
    print()
    if not item["problems"]:
        print("Problems:    none")
    for p in item["problems"]:
        print("Problem:    ", p)  # one check from scenarios.json that failed

First scenario: S3a, a late order that needs a refund above the approval limit.

In [5]:
# Scenario S3a: the late espresso machine, a $250 refund that needs approval.
# run_scenario plays the scenario's turns against a fresh agent and returns the scored result as a dict.
result = await run_scenario(LangGraphShipFast(), scenarios["S3a_late_refund_approved"])
show(result)

Scenario:    S3a_late_refund_approved
Result:      PASS
Cost:        6 model calls in 53s
Tools used:  lookup_order, track_shipment, search_policy, issue_refund

Turn  Intent          Order  Action       Refund
1     refund_request  A1003  refunded      250.0
      reply: Your refund for order A1003 has been issued. You will receive a full refund of $250.00. Thank you for your patience and understanding.

Problems:    none


S3a is the late espresso machine, a $250 refund the supervisor approves. A passing run pauses at the
approval gate, is resumed with `approve`, and issues exactly $250.

In [6]:
# Scenario S4: the broken vase, a two-turn conversation.
# run_scenario plays the scenario's turns against a fresh agent and returns the scored result as a dict.
result = await run_scenario(LangGraphShipFast(), scenarios["S4_damaged_needs_photo_then_refund"])
show(result)

Scenario:    S4_damaged_needs_photo_then_refund
Result:      PASS
Cost:        9 model calls in 75s
Tools used:  lookup_order, track_shipment, search_policy, issue_refund

Turn  Intent          Order  Action       Refund
1     complaint       A1004  needs_info        -
      reply: The vase set from your order A1004 arrived broken. According to our policy, you must provide a photo of the damage within 7 days of the delivery date. Since the
2     complaint       A1004  refunded       60.0
      reply: Your full refund of $60.00 for the broken vase set in order A1004 has been processed. You should see the refund reflected in your account within a few business 

Problems:    none


S4 is the broken vase: the first turn must ask for a photo and refund nothing (policy P3); the second,
which never names the order, must refund the $60 total. It needs memory and policy in the right order.

When we ran them, both passed. S3a paused at the approval middleware, was resumed with `approve`, and issued
exactly $250 (the runner records each turn's final state, so the pause itself is not listed as a turn).
S4 asked for the photo and refunded nothing on the first turn, then refunded $60 once on the second,
using the conversation the checkpointer kept.

## One research question

The research agent is the workflow from notebook 05, answered through the shared `run_question`. Q07 has
a stale article (9 provinces) beside the current one (14 provinces); mentioning the stale figure at all
fails the answer.

In [7]:
# Q07 has a stale article beside the current one. `answer` is this tutorial's research workflow, and
# run_question runs it on one question and returns the scored result as a dict.
item = await run_question(answer, questions["Q07_deployment"])

The result is a dict. Print its fields with a label each.

In [8]:
# Print the parts of the result worth reading, one labelled line each.
print("Passed:      ", item["passed"])  # True when the answer holds every expected fact and no stale one
print("Tools used:  ", ", ".join(item["tool_calls"]) or "none")
print("Model calls: ", item["llm_calls"])
print("Time:        ", f"{item['latency_s']:.0f}s")
print("Answer:      ", item["report"]["answer"])
print("Problems:    ", "; ".join(item["problems"]) or "none")  # what the scorer objected to, if anything

Passed:       True
Tools used:   web_search, web_search, web_search
Model calls:  7
Time:         126s
Answer:       Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025, as reported in their 2025 annual report [F04].
Problems:     none


## The full evaluation

`run_eval.py` runs every scenario and every question once and writes `results/shipfast/langgraph.json` and
`results/research/langgraph.json` (from the tutorial folder: `uv run python run_eval.py`). It takes about half an
hour on a 16 GB M4 Mac, so here we read the saved results instead of re-running them.

The table below has one row per scenario or question: its id, whether it passed, and how many seconds it took. Under a failing row, each `problem` is one check the scorer reported.

In [9]:
import json
from pathlib import Path

# The saved results live in results/<agent>/<framework>.json at the repo root, one level above the tutorial folder.
RESULTS = Path("..").resolve().parent / "results"


# Read one results file; returns None if that framework has not been run yet.
def load(agent, framework):
    path = RESULTS / agent / f"{framework}.json"
    return json.loads(path.read_text()) if path.exists() else None

In [10]:
# One block per agent: a summary line, then a table with one row per scenario or question.
for agent, key in (("shipfast", "scenarios"), ("research", "questions")):
    data = load(agent, "langgraph")
    summary = data["summary"]
    print(f"{agent}: {summary['passed']} of {summary['total']} passed (run at {data['run_at']})")
    print(f"  {'id':40} {'result':6} {'seconds':>7}")
    for it in data[key]:
        print(f"  {it['id']:40} {'PASS' if it['passed'] else 'FAIL':6} {it['latency_s']:7.0f}")
        for problem in it["problems"]:  # only failing items have problems
            print(f"      problem: {problem}")
    print()

shipfast: 7 of 8 passed (run at 2026-11-03T08:02:55+00:00)
  id                                       result seconds
  S1_status_with_followup                  PASS        58
  S2_lost_refund_auto                      PASS        40
  S3a_late_refund_approved                 PASS        55
  S3b_late_refund_rejected                 PASS        74
  S4_damaged_needs_photo_then_refund       PASS        93
  S5_outside_refund_window                 FAIL        44
      problem: action: expected 'declined', got 'refunded'
      problem: refund_amount: expected None, got 35.0
  S6_unknown_order                         PASS        19
  S7_out_of_scope                          PASS         4

research: 10 of 10 passed (run at 2026-11-03T08:27:30+00:00)
  id                                       result seconds
  Q01_headcount                            PASS       122
  Q02_founding                             PASS       120
  Q03_revenue_growth                       PASS       152
  Q04_series

## All three frameworks

The same contract, data, model, settings, tools, instructions, scoring and runner, so the differences
below come from the frameworks (and from how a small model reacts to each one's way of packaging the
same request).

In the table, `passed` is passed over total, `tokens` is the total tokens the model read and wrote, and `seconds` is the total run time, for that agent in that framework.

In [11]:
# Collect one row of totals per agent and framework, then print them as a table.
rows = []
for agent in ("shipfast", "research"):
    for framework in ("langgraph", "crewai", "strands"):
        data = load(agent, framework)
        if data:  # skip a framework whose results file does not exist yet
            s = data["summary"]
            rows.append((agent, framework, f"{s['passed']}/{s['total']}", s["total_tokens"], round(s["total_latency_s"])))

print(f"{'agent':9} {'framework':10} {'passed':7} {'tokens':>7} {'seconds':>8}")
for agent, framework, passed, tokens, seconds in rows:
    print(f"{agent:9} {framework:10} {passed:7} {tokens:7} {seconds:8}")

agent     framework  passed   tokens  seconds
shipfast  langgraph  7/8       29962      387
shipfast  crewai     7/8       34867      401
shipfast  strands    4/8       26981      331
research  langgraph  10/10     44361     1464
research  crewai     9/10      60243     1194
research  strands    9/10      48803     1147


LangGraph passed 7 of 8 ShipFast scenarios and all 10 research questions. Its one ShipFast failure, S5,
refunded the stand delivered 46 days ago: the 30-day rule is in the policy the model reads, but nothing in
code enforces it, and the model missed it, as it did in notebook 03.

Across the frameworks, LangGraph and CrewAI tie on ShipFast with different single failures; Strands is
lower, mostly because its agent twice announced the $250 refund and ended its turn without attempting it,
so the approval gate never had anything to pause. On tokens, CrewAI used the most for both agents; Strands used the fewest
on ShipFast and LangGraph the fewest on research; LangGraph took the longest on research. These are single runs on one laptop: a first
measurement, not a ranking.

## How the others do this

- **CrewAI** builds the same agent as a **Flow**, one run per customer turn with the history in the flow's
  state; the approval pause sits between flow steps and resumes from saved state.
- **Strands** uses **one agent per conversation**, with triage as a separate structured call and the
  approval as a hook that interrupts before the refund.

## Takeaways

- The contract keeps the comparison fair: one copy of the data, tools, instructions, scoring and runner.
- Code records the facts (refunds, tickets, pauses); the model writes the reply and makes one small call.
- Every framework could build both agents, but each needed its own shape for the approval pause and for
  memory, and each made different mistakes with the same model. Those differences are the comparison.